# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.


In [2]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents, fetch_via_jina
from openai import OpenAI

In [3]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if api_key and api_key.startswith("sk-proj-") and len(api_key) > 10:
    print("API key looks good so far")
else:
    print(
        "There might be a problem with your API key? Please visit the troubleshooting notebook!"
    )

MODEL = "gpt-5-nano"
openai = OpenAI()

API key looks good so far


In [7]:
links = fetch_website_links("https://tiki.vn/")
links

['https://tiki.vn/khuyen-mai/ngay-hoi-freeship?from=inline_banner',
 '/',
 '#',
 '/dien-gia-dung/c1882?from=header_keyword',
 '/o-to-xe-may-xe-dap/c8594?from=header_keyword',
 '/do-choi-me-be/c2549?from=header_keyword',
 '/lam-dep-suc-khoe/c1520?from=header_keyword',
 '/nha-cua-doi-song/c1883?from=header_keyword',
 '/nha-sach-tiki/c8322?from=header_keyword',
 '/the-thao-da-ngoai/c1975?from=header_keyword',
 'https://tiki.vn/thong-tin/tiki-doi-tra-de-dang-an-tam-mua-sam',
 '/nha-sach-tiki/c8322',
 '/nha-cua-doi-song/c1883',
 '/dien-thoai-may-tinh-bang/c1789',
 '/do-choi-me-be/c2549',
 '/thiet-bi-kts-phu-kien-so/c1815',
 '/dien-gia-dung/c1882',
 '/lam-dep-suc-khoe/c1520',
 '/o-to-xe-may-xe-dap/c8594',
 '/thoi-trang-nu/c931',
 '/bach-hoa-online/c4384',
 '/the-thao-da-ngoai/c1975',
 '/thoi-trang-nam/c915',
 '/cross-border-hang-quoc-te/c17166',
 '/laptop-may-vi-tinh-linh-kien/c1846',
 '/giay-dep-nam/c1686',
 '/dien-tu-dien-lanh/c4221',
 '/giay-dep-nu/c1703',
 '/may-anh/c1801',
 '/phu-kien-t

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.

It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.


In [8]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [9]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [10]:
print(get_links_user_prompt("https://tiki.vn/"))


Here is the list of links on the website https://tiki.vn/ -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://tiki.vn/khuyen-mai/ngay-hoi-freeship?from=inline_banner
/
#
/dien-gia-dung/c1882?from=header_keyword
/o-to-xe-may-xe-dap/c8594?from=header_keyword
/do-choi-me-be/c2549?from=header_keyword
/lam-dep-suc-khoe/c1520?from=header_keyword
/nha-cua-doi-song/c1883?from=header_keyword
/nha-sach-tiki/c8322?from=header_keyword
/the-thao-da-ngoai/c1975?from=header_keyword
https://tiki.vn/thong-tin/tiki-doi-tra-de-dang-an-tam-mua-sam
/nha-sach-tiki/c8322
/nha-cua-doi-song/c1883
/dien-thoai-may-tinh-bang/c1789
/do-choi-me-be/c2549
/thiet-bi-kts-phu-kien-so/c1815
/dien-gia-dung/c1882
/lam-dep-suc-khoe/c1520
/o-to-xe-may-xe-dap/c8594
/thoi-trang-nu/c931
/bach-hoa-online/c4384
/the-thao-da-ngoai/c1975
/thoi-tr

In [11]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)},
        ],
        response_format={"type": "json_object"},
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [12]:
select_relevant_links("https://tiki.vn/")

{'links': [{'type': 'about page',
   'url': 'https://tiki.vn/thong-tin/gioi-thieu-ve-tiki'},
  {'type': 'careers page', 'url': 'https://tuyendung.tiki.vn/'},
  {'type': 'blog', 'url': 'https://tiki.vn/blog/'},
  {'type': 'app store (iOS)',
   'url': 'https://itunes.apple.com/vn/app/id958100553'},
  {'type': 'google play',
   'url': 'https://play.google.com/store/apps/details?id=vn.tiki.app.tikiandroid'},
  {'type': 'social media (Facebook)',
   'url': 'https://www.facebook.com/tiki.vn/'},
  {'type': 'video channel (YouTube)',
   'url': 'https://www.youtube.com/user/TikiVBlog'},
  {'type': 'TikiNOW service', 'url': 'https://tikinow.vn?src=footer'},
  {'type': 'TikiNOW business page',
   'url': 'https://www.tikinow.biz/%C4%91i%E1%BB%83u-kho%E1%BA%A3n-v%E1%BA%ADn-chuy%E1%BB%83n'},
  {'type': 'corporate governance',
   'url': 'https://tiki.vn/quy-che-hoat-dong-sgdtmdt'}]}

In [13]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)},
        ],
        response_format={"type": "json_object"},
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [14]:
select_relevant_links("https://tiki.vn/")

Selecting relevant links for https://tiki.vn/ by calling gpt-5-nano
Found 3 relevant links


{'links': [{'type': 'about page',
   'url': 'https://tiki.vn/thong-tin/gioi-thieu-ve-tiki'},
  {'type': 'blog', 'url': 'https://tiki.vn/blog/'},
  {'type': 'careers page', 'url': 'https://tuyendung.tiki.vn/'}]}

In [15]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 3 relevant links


{'links': [{'type': 'enterprise page',
   'url': 'https://huggingface.co/enterprise'},
  {'type': 'company page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano


In [18]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_via_jina(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links["links"]:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_via_jina(link["url"])
    return result

In [19]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

[![Image 1: Hugging Face's logo](https://huggingface.co/front/assets/huggingface_logo-noborder.svg)Hugging Face](https://huggingface.co/)

*   [Models](https://huggingface.co/models)
*   [Datasets](https://huggingface.co/datasets)
*   [Spaces](https://huggingface.co/spaces)
*    Community  
*   [Docs](https://huggingface.co/docs)
*   [Enterprise](https://huggingface.co/enterprise)
*   [Pricing](https://huggingface.co/pricing)
*    
*   
* * *

*   [Log In](https://huggingface.co/login)
*   [Sign Up](https://huggingface.co/join)

[NEW * GGML and llama.cpp join Hugging Face 🔥 * Try HuggingChat Omni – Chat with AI 💬 * Get started with Inference in seconds 🚀](https://huggingface.co/blog/ggml-joins-hf "GGML and llama.cpp join Hugging Face 🔥")

![Image 2](https://huggingface.co/front/assets/huggingface_logo-noborder.svg)
The

In [ ]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brospective customers, ochure about the company for prinvestors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

In [21]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000]  # Truncate if more than 5,000 characters
    return user_prompt

In [22]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 14 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n===============\n\n[![Image 1: Hugging Face\'s logo](https://huggingface.co/front/assets/huggingface_logo-noborder.svg)Hugging Face](https://huggingface.co/)\n\n*   [Models](https://huggingface.co/models)\n*   [Datasets](https://huggingface.co/datasets)\n*   [Spaces](https://huggingface.co/spaces)\n*    Community  \n*   [Docs](https://huggingface.co/docs)\n*   [Enterprise](https://huggingface.co/enterprise)\n*   [Pricing](https://huggingface.co/pricing)\n*    \n*   \n* * *\n\n*   [Log In](https://huggingface.co/login)\n*   [Sign Up](https://huggingface.co/join)\n\n[NEW * GGML and llama.cpp join Hugging Face 🔥 * Try HuggingChat Omni – Chat with AI 💬 * Get started with Inference in seconds 🚀](http

In [23]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)},
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [24]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 19 relevant links


# Hugging Face – The AI Community Building the Future

![Hugging Face Logo](https://huggingface.co/front/assets/huggingface_logo-noborder.svg)

---

## About Us

Hugging Face is a vibrant, global AI community and platform dedicated to advancing machine learning technology and collaboration. We empower developers, researchers, and enterprises by providing seamless access to an extensive repository of over 2 million machine learning models, datasets, and AI applications.

Our mission is to build the future of AI together, fostering innovation through open collaboration and transparent sharing of resources.

---

## Our Platform

- **Models:** Explore and utilize over 2 million state-of-the-art AI models covering a wide range of tasks and domains.
- **Datasets:** Access diverse datasets curated to fuel machine learning research and development.
- **Spaces:** Discover and deploy AI-powered applications created by the community or develop your own in an interactive environment.
- **Community:** Join a thriving community where AI practitioners exchange ideas, contribute models, datasets, and tools, fostering growth and collaboration.
- **Documentation:** Comprehensive guides and tutorials to help developers and enterprises get started and scale using Hugging Face technologies.

Explore AI apps or browse models here: [Explore AI Apps](https://huggingface.co/spaces) | [Browse Models](https://huggingface.co/models)

---

## Enterprise Solutions

Hugging Face offers tailored enterprise solutions that enable organizations to scale AI adoption efficiently. Our platform gives businesses access to cutting-edge models and tooling designed for robust deployment and integration in production environments.

Partner with us to leverage the world’s leading AI platform to accelerate innovation, enhance product offerings, and achieve business goals.

Learn more: [Enterprise Hub](https://huggingface.co/enterprise)

---

## Our Culture

We are a community-first company that thrives on openness, collaboration, and shared growth. Our culture embraces:

- **Inclusivity:** Welcoming AI enthusiasts from all backgrounds to contribute and learn.
- **Innovation:** Driving forward the frontier of AI research and applications.
- **Transparency:** Open exchange of knowledge through open-source projects and community engagement.
- **Support:** Providing extensive documentation, tools, and resources to empower users at every skill level.

---

## Join Our Team

We are continuously expanding and looking for passionate AI researchers, engineers, product managers, and community builders who want to shape the future of machine learning. At Hugging Face, you will work alongside some of the brightest minds in AI and contribute to projects with global impact.

Explore career opportunities and join our mission: [Sign Up & Careers](https://huggingface.co/join)

---

## Why Choose Hugging Face?

- Community-driven development with millions of models and datasets.
- Cutting-edge AI technology and tools accessible to everyone.
- A supportive ecosystem for collaboration and innovation.
- Scalable enterprise solutions for organizations of all sizes.

---

**Connect with Hugging Face – where the AI community builds the future.**

[Visit Our Website](https://huggingface.co/) | [Explore Models](https://huggingface.co/models) | [Discover Datasets](https://huggingface.co/datasets) | [Join the Community](https://huggingface.co/join)

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation


In [25]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)},
        ],
        stream=True,
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        update_display(Markdown(response), display_id=display_handle.display_id)

In [26]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


# Hugging Face  
*The AI community building the future.*

---

## About Us  
Hugging Face is the leading platform where the global machine learning community collaborates to build, share, and deploy models, datasets, and AI applications. With a vibrant ecosystem, we empower developers, researchers, and enterprises to accelerate AI innovation and adoption.

---

## What We Offer  

- **Models**  
  Explore over 2 million state-of-the-art machine learning models, updated regularly by contributors worldwide. Popular models include Qwen, NVIDIA’s Personaplex, MiniMaxAI, and many more.

- **Datasets**  
  Access and share high-quality datasets essential for training and benchmarking AI models.

- **Spaces**  
  Discover and build interactive AI applications using open-source resources and hosted environments.

- **Community**  
  Join a thriving community of AI practitioners collaborating, sharing knowledge, and advancing the field together.

- **Documentation**  
  Comprehensive guides and tutorials to help users seamlessly integrate Hugging Face tools and models into their projects.

- **Enterprise Solutions**  
  Customized AI platforms and support for businesses to leverage cutting-edge machine learning technologies.

- **Pricing**  
  Flexible pricing plans catering to individuals, startups, and companies of all sizes.

---

## Our Culture  
Hugging Face fosters an open and inclusive environment where transparency, collaboration, and shared learning are core values. The company thrives on community contributions and encourages innovation with a strong focus on ethical AI development. As a workplace, Hugging Face values continuous growth, creativity, and impact-driven work.


---

## Customers and Partners  
Hugging Face serves a broad spectrum of users ranging from individual developers and researchers to global tech leaders and enterprises. Prominent AI organizations and companies like NVIDIA contribute and use Hugging Face’s models and infrastructure — demonstrating trust and industry standard-setting influence.

---

## Careers  
Join Hugging Face to be at the forefront of AI technology! The company offers opportunities in research, engineering, product development, community management, and more. Employees enjoy a dynamic and collaborative workplace focused on meaningful AI advancements that benefit everyone.

- [Explore Current Openings](https://huggingface.co/join)  
- Embrace a mission-driven company that values diversity and creativity  
- Work alongside top AI experts and a vibrant international community

---

## Get Started  

- Visit [huggingface.co](https://huggingface.co/) to explore models, datasets, and AI apps.  
- Join the AI revolution with easy-to-use tools and comprehensive resources.  
- Sign up for free to start building or integrating AI solutions in seconds.

---

### Hugging Face – The AI community building the future.  
Collaborate, innovate, and power your AI projects with the world’s leading ML platform.  

[Explore Models](https://huggingface.co/models) | [Browse Datasets](https://huggingface.co/datasets) | [Discover Spaces](https://huggingface.co/spaces) | [Join Community](https://huggingface.co/join)

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>

</td>
</tr>

</table>


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>
